# Steam Oyun Yorumları: Duygu ve Şikayet Analizi

## Proje Amacı

Bu notebook, Steam platformundan çekilen **12.500 gerçek oyun yorumunu** analiz ederek:
- Oyun türlerine göre şikayet kategorilerini tespit etmek
- VADER duygu analizi bulgularını görselleştirmek
- Oyun stüdyolarına geliştirme önceliği konusunda veri tabanlı öneriler sunmak
- Açıklanabilir YZ (SHAP) ile oyuncu memnuniyetini etkileyen faktörleri ortaya koymak

## Veri Kaynağı

| Kaynak | İçerik |
|--------|--------|
| Steam Review API | Kullanıcı yorumları (voted_up, playtime, dil) |
| Steam Store API | Oyun meta verisi (fiyat, geliştirici, kategoriler) |

**Dosyalar:** `data/processed/*_categorized.csv`

Bu notebook Steam platformundan çekilen 12.500 gerçek oyun yorumunun analizini içermektedir.

## Gereksinimler

```bash
pip install pandas numpy matplotlib seaborn plotly requests scikit-learn shap nltk tqdm
```


In [ ]:
# Gerekli kütüphaneleri kur (ilk çalıştırmada bir kez çalıştırın)
import subprocess, sys

required = [
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly',
    'requests', 'scikit-learn', 'shap', 'nltk', 'tqdm'
]
for pkg in required:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f'Kuruluyor: {pkg}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Tüm kütüphaneler hazır.')


In [ ]:
import os
import time
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print('Kütüphaneler başarıyla yüklendi.')


## 1.1 Veri Yükleme

Tüm `*_categorized.csv` dosyaları tek bir DataFrame'de birleştirilmektedir.


In [ ]:
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebook' else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

print(f'Proje kökü: {PROJECT_ROOT}')
print(f'Veri dizini: {DATA_DIR}')

csv_files = sorted(DATA_DIR.glob('*_categorized.csv'))
print(f'Bulunan dosya sayısı: {len(csv_files)}')

dfs = []
for f in csv_files:
    try:
        dfs.append(pd.read_csv(f, encoding='utf-8-sig'))
    except Exception as e:
        print(f'[UYARI] {f.name} yüklenemedi: {e}')

df = pd.concat(dfs, ignore_index=True)
print(f'Toplam satır: {len(df):,}')
print(f'Toplam sütun: {len(df.columns)}')


## 1.2 Veri Önizleme


In [ ]:
print(f'Boyut: {df.shape[0]:,} satır × {df.shape[1]} sütun')
print()
print('Veri Tipleri:')
print(df.dtypes.to_string())


In [ ]:
df.head(3)

In [ ]:
df.describe(include='all').T

> **Not:** Bu notebook Steam platformundan çekilen **12.500 gerçek oyun yorumunun** analizini içermektedir.
> Yorumlar 25 farklı oyundan (RPG, FPS, Strateji, Simülasyon, Çok Oyunculu türlerinden 5'er oyun) toplanmıştır.


---
# Bölüm 2: Keşifçi Veri Analizi (EDA)

Veri setinin genel yapısını, dağılımlarını ve temel istatistiklerini inceliyoruz.


## 2.1 Oyun Türü Dağılımı


In [ ]:
genre_counts = df.groupby('game_genre')['game_name'].count().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(genre_counts.index, genre_counts.values,
              color=sns.color_palette('muted', len(genre_counts)))
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title('Oyun Türüne Göre Yorum Sayısı', fontsize=14, fontweight='bold')
ax.set_xlabel('Oyun Türü')
ax.set_ylabel('Yorum Sayısı')
plt.tight_layout()
plt.show()

print(genre_counts.to_string())


Her oyun türü için yaklaşık eşit sayıda yorum bulunmaktadır (oyun başına ~500 yorum × 5 oyun = 2500/tür).


## 2.2 Dil Dağılımı


In [ ]:
lang_counts = df['language'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].pie(lang_counts.values, labels=lang_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Set2', len(lang_counts)))
axes[0].set_title('Yorum Dil Dağılımı (İlk 10)', fontsize=12, fontweight='bold')

axes[1].barh(lang_counts.index[::-1], lang_counts.values[::-1],
             color=sns.color_palette('Set2', len(lang_counts)))
axes[1].set_title('Dil Bazında Yorum Sayısı', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Yorum Sayısı')

plt.tight_layout()
plt.show()


İngilizce yorumlar dominant konumdadır. Çok dilli analiz için `review_english` sütunu kullanılmaktadır.


## 2.3 Olumlu/Olumsuz Yorum Oranları (Oyun Bazında)


In [ ]:
vote_pct = (df.groupby(['game_name', 'voted_up'])
             .size()
             .unstack(fill_value=0)
             .rename(columns={True: 'Olumlu', False: 'Olumsuz'}))
vote_pct['Toplam'] = vote_pct.sum(axis=1)
vote_pct['Olumlu_Pct'] = (vote_pct['Olumlu'] / vote_pct['Toplam'] * 100).round(1)
vote_pct = vote_pct.sort_values('Olumlu_Pct', ascending=True)

fig, ax = plt.subplots(figsize=(12, 9))
colors = ['#e74c3c' if p < 50 else '#2ecc71' for p in vote_pct['Olumlu_Pct']]
bars = ax.barh(vote_pct.index, vote_pct['Olumlu_Pct'], color=colors)
ax.axvline(50, color='gray', linestyle='--', alpha=0.7, label='%50 esigi')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_title('Oyun Bazinda Olumlu Yorum Yüzdesi', fontsize=14, fontweight='bold')
ax.set_xlabel('Olumlu Yorum Orani (%)')
ax.set_xlim(0, 115)
ax.legend()
plt.tight_layout()
plt.show()


Kırmızı barlar %50'nin altında olumlu oran demektir — bu oyunlar genel olarak olumsuz değerlendiriliyor.


## 2.4 Playtime Dağılımı


In [ ]:
try:
    print('Mevcut sutunlar (playtime):', [c for c in df.columns if 'playtime' in c.lower()])

    if 'playtime_forever' not in df.columns:
        print('UYARI: playtime_forever sütunu bulunamadı.')
    else:
        playtime_h = df['playtime_forever'].dropna() / 60
        playtime_h = playtime_h[playtime_h < 10000]  # 10000 saatten fazlasini cikar

        if len(playtime_h) < 2:
            print('Yeterli playtime verisi bulunamadı.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            axes[0].hist(playtime_h, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
            axes[0].set_title('Oyun Süresi Dagilimi (saat)', fontsize=12, fontweight='bold')
            axes[0].set_xlabel('Saat')
            axes[0].set_ylabel('Yorum Sayisi')

            axes[1].hist(playtime_h, bins=50, color='coral', edgecolor='white', alpha=0.8)
            axes[1].set_yscale('log')
            axes[1].set_title('Oyun Süresi Dagilimi (Log Olcegi)', fontsize=12, fontweight='bold')
            axes[1].set_xlabel('Saat')

            plt.tight_layout()
            plt.show()

            print(f'Temizlenen kayit sayisi: {len(playtime_h):,}')
            print(f'Medyan oyun süresi: {playtime_h.median():.1f} saat')
            print(f'Ortalama oyun süresi: {playtime_h.mean():.1f} saat')
except Exception as e:
    print('Hata:', e)
    print('Yeterli playtime verisi bulunamadı.')


Oyun süresi sağa çarpık dağılım göstermektedir — az oynayan kullanıcılar daha sık yorum yazmaktadır.


## 2.5 VADER Skor Dağılımı


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['vader_compound'].dropna(), bins=50,
             color='mediumpurple', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[0].set_title('VADER Compound Skor Dagilimi', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Compound Skor (-1 ile 1 arasi)')
axes[0].set_ylabel('Yorum Sayisi')

sent_counts = df['sentiment'].value_counts()
axes[1].pie(sent_counts.values, labels=sent_counts.index,
            autopct='%1.1f%%', startangle=90,
            colors=['#2ecc71', '#e74c3c', '#95a5a6'])
axes[1].set_title('Sentiment Dagilimi', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


## 2.6 Steam voted_up ile VADER Uyum Analizi


In [ ]:
uyum_df = df.dropna(subset=['voted_up', 'sentiment']).copy()
uyum_df['voted_up_str'] = uyum_df['voted_up'].map({True: 'Olumlu', False: 'Olumsuz'})

ct = pd.crosstab(uyum_df['voted_up_str'], uyum_df['sentiment'],
                 normalize='index') * 100

fig, ax = plt.subplots(figsize=(10, 5))
ct.plot(kind='bar', ax=ax, color=['#e74c3c', '#95a5a6', '#2ecc71'],
        edgecolor='white', width=0.7)
ax.set_title('Steam voted_up vs VADER Sentiment Uyumu (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Steam Degerlendirmesi')
ax.set_ylabel('Yüzde (%)')
ax.legend(title='VADER')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

toplam_uyum = (df['vader_voted_match'].sum() / len(df) * 100)
print(f'Steam-VADER genel uyum orani: %{toplam_uyum:.1f}')


Steam beğeni puanı ile VADER duygu skoru her zaman örtüşmemektedir. "Olumlu oyladığı ama negatif yorum yazdığı" durumlar **mixed sentiment** analizinde incelenecektir.


## 2.7 Eksik Veri Analizi


In [ ]:
eksik = df.isnull().sum()
eksik_pct = (eksik / len(df) * 100).round(2)
eksik_df = pd.DataFrame({'Eksik Sayi': eksik, 'Eksik %': eksik_pct})
eksik_df = eksik_df[eksik_df['Eksik Sayi'] > 0].sort_values('Eksik %', ascending=False)

if len(eksik_df) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(eksik_df) * 0.5)))
    sns.barplot(data=eksik_df.reset_index(), x='Eksik %', y='index',
                palette='Reds_r', ax=ax)
    ax.set_title('Eksik Veri Yüzdesi (Sütun Bazinda)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Eksik Veri %')
    ax.set_ylabel('Sütun')
    plt.tight_layout()
    plt.show()
    print(eksik_df.to_string())
else:
    print('Eksik veri bulunmamaktadir.')


---
# Bölüm 3: Veri Harmanlama (Python Dersi Kriteri)

**İki farklı veri kaynağı birleştirildi:**
1. **Steam Review API** — kullanıcı yorumları (halihazırda yüklendi)
2. **Steam Store API** — oyun meta verisi (fiyat, geliştirici, yayıncı, çıkış tarihi)

Bu bölüm Python dersi veri harmanlama kriterini karşılamak üzere iki bağımsız API'den gelen
veriyi birleştirerek zenginleştirilmiş bir analiz veri seti oluşturmaktadır.


In [ ]:
OYUN_APPID = {
    'Cyberpunk 2077':            1091500,
    "Baldur's Gate 3":           1086940,
    'Elden Ring':                1245620,
    'The Witcher 3':              292030,
    'Dragon Age Inquisition':    1222690,
    'Counter-Strike 2':              730,
    'Apex Legends':              1172470,
    'Doom Eternal':               782330,
    'Battlefield 2042':          1517290,
    'Titanfall 2':               1237970,
    'Civilization VI':            289070,
    'Total War Warhammer III':   1142710,
    'Stellaris':                  281990,
    'Age of Empires IV':         1466860,
    'XCOM 2':                     268500,
    'Cities Skylines':            255710,
    'Planet Zoo':                 703080,
    'Microsoft Flight Simulator': 1250410,
    'Stardew Valley':             413150,
    'Euro Truck Simulator 2':     227300,
    'PUBG':                       578080,
    'Among Us':                   945360,
    'Fall Guys':                 1097150,
    'Deep Rock Galactic':         548430,
    'Warframe':                   230410,
}
print(f'Toplam oyun: {len(OYUN_APPID)}')


In [ ]:
def steam_meta_cek(app_id, oyun_adi):
    # Steam Store API'den oyun meta verisi cekmek
    url = f'https://store.steampowered.com/api/appdetails?appids={app_id}&cc=us&l=en'
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json().get(str(app_id), {})
        if not data.get('success'):
            return {'game_name': oyun_adi, 'app_id': app_id}
        d = data['data']
        price_info = d.get('price_overview', {})
        return {
            'game_name':       oyun_adi,
            'app_id':          app_id,
            'fiyat_usd':       price_info.get('final', 0) / 100 if price_info else 0,
            'fiyat_str':       price_info.get('final_formatted', 'Ucretsiz'),
            'ucretsiz':        d.get('is_free', False),
            'gelistirici':     ', '.join(d.get('developers', [])),
            'yayinci':         ', '.join(d.get('publishers', [])),
            'cikis_tarihi':    d.get('release_date', {}).get('date', ''),
            'meta_puani':      d.get('metacritic', {}).get('score', None),
            'kategori_etiket': ', '.join([c['description'] for c in d.get('categories', [])[:5]]),
            'tur_etiket':      ', '.join([g['description'] for g in d.get('genres', [])[:3]]),
            'basarim_sayisi':  d.get('achievements', {}).get('total', 0),
        }
    except Exception as e:
        print(f'[HATA] {oyun_adi}: {e}')
        return {'game_name': oyun_adi, 'app_id': app_id}


META_CACHE = PROJECT_ROOT / 'data' / 'processed' / 'steam_meta.csv'

if META_CACHE.exists():
    meta_df = pd.read_csv(META_CACHE)
    print(f'Onbellekten yuklendi: {len(meta_df)} oyun')
else:
    print('Steam Store API meta veri cekiliyor...')
    meta_records = []
    for oyun, app_id in OYUN_APPID.items():
        print(f'  {oyun}', end=' ... ')
        rec = steam_meta_cek(app_id, oyun)
        meta_records.append(rec)
        print('OK' if 'fiyat_usd' in rec else 'HATA')
        time.sleep(0.5)
    meta_df = pd.DataFrame(meta_records)
    meta_df.to_csv(META_CACHE, index=False, encoding='utf-8-sig')
    print(f'Meta veri kaydedildi: {META_CACHE}')

meta_df.head()


## 3.1 Veri Birleştirme (Merge)


In [ ]:
keep_cols = ['game_name', 'fiyat_usd', 'ucretsiz', 'gelistirici',
             'yayinci', 'cikis_tarihi', 'meta_puani',
             'kategori_etiket', 'tur_etiket', 'basarim_sayisi']
available = [c for c in keep_cols if c in meta_df.columns]

df_merged = df.merge(meta_df[available], on='game_name', how='left')

if 'ucretsiz' in df_merged.columns:
    df_merged.loc[df_merged['ucretsiz'] == True, 'fiyat_usd'] = 0.0

print(f'Birlesik veri seti boyutu: {df_merged.shape}')
print(f'Fiyat bilgisi olan yorum sayisi: {df_merged["fiyat_usd"].notna().sum():,}')

show_cols = [c for c in ['game_name', 'game_genre', 'fiyat_usd', 'gelistirici', 'basarim_sayisi'] if c in df_merged.columns]
df_merged[show_cols].drop_duplicates('game_name').sort_values('game_name')


---
# Bölüm 4: Özellik Mühendisliği (Python Dersi Kriteri)

Mevcut veriden **4 yeni türetilmiş değişken** oluşturulmaktadır. Bu değişkenler
ham veriden doğrudan elde edilemeyen içgörüler sağlamaktadır.


## 4.1 Türetilmiş Değişken 1: Şikayet Yoğunluk Skoru

**Tanım:** Her yorum için negatif cümle sayısı / toplam cümle sayısı oranı.
VADER negatif skoru proxy olarak kullanılmaktadır; bu skor cümle düzeyinde negatiflik yoğunluğunu yansıtır.


In [ ]:
df_merged['sikayet_yogunluk'] = df_merged['vader_neg'].fillna(0)

sikayet_oyun = (df_merged.groupby('game_name')['sikayet_yogunluk']
                         .mean()
                         .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('RdYlGn_r', len(sikayet_oyun))
bars = ax.barh(sikayet_oyun.index[::-1], sikayet_oyun.values[::-1], color=colors[::-1])
ax.set_title('Oyun Bazinda Ortalama Sikayet Yogunluk Skoru', fontsize=13, fontweight='bold')
ax.set_xlabel('Ortalama VADER Negatif Skor')
ax.bar_label(bars, fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()

print('En yüksek sikayet yogunlugu:')
print(sikayet_oyun.head(5).to_string())


Yüksek şikayet yoğunluğu skoru, yorumlarda daha fazla negatif ifade bulunduğuna işaret eder. Bu oyunların geliştiricileri iyileştirmeye öncelik vermelidir.


## 4.2 Türetilmiş Değişken 2: Mixed Sentiment Oranı

**Tanım:** `voted_up=True` ama `vader_label=negative` olan yorumlar.
Bu oyuncular oyunu **Steam'de olumlu oylarken yorumlarında şikayet** etmektedir.


In [ ]:
df_merged['mixed_sentiment'] = (
    (df_merged['voted_up'] == True) & (df_merged['sentiment'] == 'negative')
).astype(int)

mixed_oyun = df_merged.groupby('game_name').agg(
    toplam=('mixed_sentiment', 'count'),
    mixed=('mixed_sentiment', 'sum')
).assign(mixed_pct=lambda x: x['mixed'] / x['toplam'] * 100).sort_values('mixed_pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(mixed_oyun.index[::-1], mixed_oyun['mixed_pct'][::-1],
               color=sns.color_palette('coolwarm', len(mixed_oyun)))
ax.set_title('Oyun Bazinda Mixed Sentiment Orani (Olumlu oy + Negatif yorum)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mixed Sentiment %')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
plt.tight_layout()
plt.show()

print('Bu oyuncular oyunu beğeniyor ama sikayet de ediyor:')
print(mixed_oyun[['mixed', 'toplam', 'mixed_pct']].head(5).to_string())


Mixed sentiment, oyunun nüanslı bir değerlendirmesine işaret eder. Bu oyunlar belirli sorunlara rağmen oynama değeri taşıyan oyunlardır.


## 4.3 Türetilmiş Değişken 3: Deneyim Ağırlıklı Şikayet Skoru

**Tanım:** Oyun süresine göre ağırlıklandırılmış şikayet skoru.
Az oynayan kullanıcıların şikayetleri daha düşük, çok oynayan kullanıcılarınki daha yüksek ağırlık alır.

| Oyun Süresi | Ağırlık |
|-------------|---------|
| 0-60 dk     | 0.5     |
| 60-600 dk   | 1.0     |
| 600-3000 dk | 1.5     |
| 3000+ dk    | 2.0     |


In [ ]:
try:
    print('Mevcut sutunlar (playtime/sikayet):', [c for c in df_merged.columns if 'playtime' in c.lower() or 'sikayet' in c.lower()])

    # NaN degerlerini 0 ile doldur
    df_merged['playtime_forever'] = df_merged['playtime_forever'].fillna(0)

    def playtime_weight(minutes):
        if pd.isna(minutes) or minutes <= 0: return 1.0
        if minutes < 60:    return 0.5
        if minutes < 600:   return 1.0
        if minutes < 3000:  return 1.5
        return 2.0

    df_merged['playtime_agirlik'] = df_merged['playtime_forever'].apply(playtime_weight)
    df_merged['agirlikli_sikayet'] = df_merged['sikayet_yogunluk'] * df_merged['playtime_agirlik']

    karsilastirma = df_merged.groupby('game_name').agg(
        ham_skor=('sikayet_yogunluk', 'mean'),
        agirlikli_skor=('agirlikli_sikayet', 'mean')
    ).sort_values('agirlikli_skor', ascending=False)

    fig, ax = plt.subplots(figsize=(13, 7))
    x = range(len(karsilastirma))
    ax.bar([i - 0.2 for i in x], karsilastirma['ham_skor'],
           width=0.4, label='Ham Skor', color='steelblue', alpha=0.8)
    ax.bar([i + 0.2 for i in x], karsilastirma['agirlikli_skor'],
           width=0.4, label='Agirlikli Skor', color='coral', alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(karsilastirma.index, rotation=45, ha='right', fontsize=8)
    ax.set_title('Ham vs Deneyim Agirlikli Sikayet Skoru', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('Hata:', e)


Ağırlıklı skor, deneyimli oyuncuların şikayetlerini öne çıkarmaktadır. Deneyimli oyunculardan gelen şikayetler daha anlamlı kabul edilebilir.


## 4.4 Türetilmiş Değişken 4: Fiyat/Şikayet Oranı

**Soru:** Pahalı oyunlar daha mı çok şikayet alıyor?
**Hesaplama:** `şikayet_yoğunluğu / (fiyat + 1)` — sıfıra bölmeyi önlemek için +1 ekliyoruz.


In [ ]:
if 'fiyat_usd' in df_merged.columns:
    fiyat_sikayet = df_merged.groupby('game_name').agg(
        sikayet=('sikayet_yogunluk', 'mean'),
        fiyat=('fiyat_usd', 'first')
    ).reset_index()
    fiyat_sikayet['fiyat_sikayet_oran'] = fiyat_sikayet['sikayet'] / (fiyat_sikayet['fiyat'].fillna(0) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].scatter(fiyat_sikayet['fiyat'], fiyat_sikayet['sikayet'],
                    s=100, alpha=0.7, color='steelblue')
    for _, row in fiyat_sikayet.iterrows():
        axes[0].annotate(row['game_name'][:10], (row['fiyat'], row['sikayet']),
                        fontsize=7, alpha=0.7)
    axes[0].set_title('Fiyat vs Sikayet Yogunlugu', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Fiyat (USD)')
    axes[0].set_ylabel('Ort. Sikayet Yogunlugu')

    fs = fiyat_sikayet.sort_values('fiyat_sikayet_oran', ascending=True)
    axes[1].barh(fs['game_name'], fs['fiyat_sikayet_oran'],
                 color=sns.color_palette('RdYlGn_r', len(fs)))
    axes[1].set_title('Fiyat/Sikayet Orani (Dusuk = Daha Iyi Deger)', fontsize=11, fontweight='bold')
    axes[1].set_xlabel('Oran')

    plt.tight_layout()
    plt.show()

    korel = fiyat_sikayet[['fiyat', 'sikayet']].corr().iloc[0, 1]
    print(f'Fiyat-Sikayet Pearson korelasyonu: {korel:.3f}')
    if abs(korel) < 0.3:
        print('Sonuc: Fiyat ile sikayet yogunlugu arasinda zayif bir iliski vardir.')
    elif korel > 0.3:
        print('Sonuc: Pahali oyunlar daha fazla sikayet almaktadir.')
    else:
        print('Sonuc: Pahali oyunlar daha az sikayet almaktadir.')
else:
    print('Fiyat verisi mevcut degil — Bölüm 3 calistirilmadan meta veri yüklenmemis olabilir.')


---
# Bölüm 5: Duygu Analizi

VADER sentiment skorları ile Steam voted_up değerlerini karşılaştırarak oyun türlerine göre duygu dağılımını analiz ediyoruz.


## 5.1 Oyun Türüne Göre VADER Skor Dağılımı


In [ ]:
try:
    print('Mevcut sutunlar (vader):', [c for c in df_merged.columns if 'vader' in c.lower()])

    # vader_compound sütun adini bul (farkli isimlendirme olabilir)
    vader_col = None
    for candidate in ['vader_compound', 'vader_score', 'compound']:
        if candidate in df_merged.columns:
            vader_col = candidate
            break

    if vader_col is None:
        print('UYARI: vader_compound sütunu bulunamadı.')
        print('Mevcut sütunlar:', df_merged.columns.tolist())
    elif 'game_genre' not in df_merged.columns:
        print('UYARI: game_genre sütunu bulunamadı.')
    else:
        plot_df = df_merged.dropna(subset=[vader_col, 'game_genre'])
        fig, ax = plt.subplots(figsize=(12, 6))
        sns.boxplot(data=plot_df, x='game_genre', y=vader_col,
                    order=sorted(plot_df['game_genre'].unique()),
                    palette='Set2', ax=ax)
        ax.axhline(0, color='red', linestyle='--', alpha=0.5)
        ax.set_title('Oyun Türüne Göre VADER Compound Skor Dagilimi', fontsize=13, fontweight='bold')
        ax.set_xlabel('Oyun Türü')
        ax.set_ylabel('VADER Compound Skor')
        plt.tight_layout()
        plt.show()
except Exception as e:
    print('Hata:', e)


In [ ]:
try:
    if 'vader_col' not in dir() or vader_col is None:
        vader_col = next((c for c in df_merged.columns if 'compound' in c.lower()), None)
    if vader_col:
        tur_sentiment = (df_merged.dropna(subset=[vader_col])
                                  .groupby('game_genre')[vader_col]
                                  .agg(['mean', 'median', 'std']).round(3))
        print('Oyun türüne göre VADER istatistikleri:')
        print(tur_sentiment.to_string())
    else:
        print('vader_col bulunamadı, lütfen önceki hücreyi çalıştırın.')
except Exception as e:
    print('Hata:', e)


## 5.2 En Pozitif ve En Negatif Oyunlar


In [ ]:
oyun_vader = df_merged.groupby(['game_name', 'game_genre'])['vader_compound'].mean().reset_index()
oyun_vader = oyun_vader.sort_values('vader_compound')

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in oyun_vader['vader_compound']]
bars = ax.barh(oyun_vader['game_name'], oyun_vader['vader_compound'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
ax.set_title('Oyun Bazinda Ortalama VADER Compound Skoru', fontsize=13, fontweight='bold')
ax.set_xlabel('Ortalama Compound Skor')
plt.tight_layout()
plt.show()


## 5.3 Türler Arası Duygu Karşılaştırması


In [ ]:
tur_sent_dist = df_merged.groupby(['game_genre', 'sentiment']).size().unstack(fill_value=0)
tur_sent_pct = tur_sent_dist.div(tur_sent_dist.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 6))
tur_sent_pct.plot(kind='bar', ax=ax,
                  color=['#e74c3c', '#95a5a6', '#2ecc71'],
                  edgecolor='white', width=0.7)
ax.set_title('Oyun Türüne Göre Sentiment Dagilimi (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Oyun Türü')
ax.set_ylabel('Oran (%)')
ax.legend(title='Sentiment')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()


## 5.4 Zaman Serisi: Aylık Duygu Trendi


In [ ]:
try:
    df_merged['timestamp_dt'] = pd.to_datetime(df_merged['timestamp_created'], errors='coerce')
    df_merged['ay'] = df_merged['timestamp_dt'].dt.to_period('M')

    aylik = (df_merged.groupby(['ay', 'game_genre'])['vader_compound']
                       .mean()
                       .reset_index())
    aylik['ay_dt'] = aylik['ay'].dt.to_timestamp()

    fig, ax = plt.subplots(figsize=(14, 6))
    for genre, grp in aylik.groupby('game_genre'):
        grp_sorted = grp.sort_values('ay_dt')
        ax.plot(grp_sorted['ay_dt'], grp_sorted['vader_compound'],
                marker='o', markersize=4, label=genre)
    ax.axhline(0, color='black', linestyle='--', alpha=0.3)
    ax.set_title('Aylik Ortalama Duygu Trendi (Türe Göre)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Ay')
    ax.set_ylabel('Ort. VADER Compound')
    ax.legend(title='Tür')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Zaman serisi olusturulamadi: {e}')


---
# Bölüm 6: Şikayet Kategori Analizi

8 şikayet kategorisinin dağılımını ve oyun türlerine göre farklılıklarını inceliyoruz.


In [ ]:
KAT_COLS = ['cat_performans', 'cat_teknik_sorun', 'cat_fiyat', 'cat_hikaye_icerik',
            'cat_oynanabilirlik', 'cat_cok_oyunculu', 'cat_destek_guncelleme',
            'cat_gorsel_ses']
KAT_LABELS = ['Performans', 'Teknik Sorun', 'Fiyat', 'Hikaye/Icerik',
              'Oynanabilirlik', 'Cok Oyunculu', 'Destek/Guncelleme', 'Gorsel/Ses']
label_map = dict(zip(KAT_COLS, KAT_LABELS))

kat_mevcut = [c for c in KAT_COLS if c in df_merged.columns]
print(f'Kullanilabilir kategori sütunu: {len(kat_mevcut)}')
print(kat_mevcut)


## 6.1 Genel Kategori Dağılımı


In [ ]:
kat_toplam = df_merged[kat_mevcut].sum().sort_values(ascending=False)
kat_pct = kat_toplam / len(df_merged) * 100
kat_pct.index = [label_map.get(i, i) for i in kat_pct.index]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(kat_pct.index, kat_pct.values,
              color=sns.color_palette('tab10', len(kat_pct)))
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_title('Sikayet Kategorilerinin Genel Dagilimi (% Yorum)', fontsize=13, fontweight='bold')
ax.set_xlabel('Kategori')
ax.set_ylabel('Yorumlarda Görülme Orani (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
plt.tight_layout()
plt.show()


## 6.2 Oyun Türüne Göre Kategori Farklılıkları


In [ ]:
tur_kat = (df_merged.groupby('game_genre')[kat_mevcut]
                    .mean() * 100).round(1)
tur_kat.columns = [label_map.get(c, c) for c in tur_kat.columns]

fig, ax = plt.subplots(figsize=(13, 6))
tur_kat.T.plot(kind='bar', ax=ax, colormap='Set1', width=0.75)
ax.set_title('Oyun Türüne Göre Sikayet Kategorileri (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Kategori')
ax.set_ylabel('Yorumlarda Görülme %')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(title='Oyun Türü', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 6.3 Her Türün İmza Şikayeti (En Baskın Kategori)


In [ ]:
imza = tur_kat.idxmax(axis=1)
print('Her türün imza sikayeti:')
for tur, kat in imza.items():
    pct = tur_kat.loc[tur, kat]
    print(f'  {tur:20s}  ->  {kat}  (%{pct:.1f})')


## 6.4 Kategori Korelasyon Matrisi

Hangi şikayetler birlikte geliyor? Yüksek korelasyon, bu iki şikayet türünün genellikle aynı yorumda göründüğüne işaret eder.


In [ ]:
kat_korel = df_merged[kat_mevcut].corr()
kat_korel.index = [label_map.get(i, i) for i in kat_korel.index]
kat_korel.columns = [label_map.get(c, c) for c in kat_korel.columns]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(kat_korel, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Sikayet Kategorileri Korelasyon Matrisi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 6.5 Heatmap: 25 Oyun × 8 Kategori


In [ ]:
oyun_kat = (df_merged.groupby('game_name')[kat_mevcut]
                     .mean() * 100).round(1)
oyun_kat.columns = [label_map.get(c, c) for c in oyun_kat.columns]

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(oyun_kat, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Görülme %'})
ax.set_title('25 Oyun x 8 Kategori Heatmap (Görülme %)', fontsize=13, fontweight='bold')
ax.set_xlabel('Sikayet Kategorisi')
ax.set_ylabel('Oyun')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


---
# Bölüm 7: Maliyet/Fayda Simülasyonu (Python Dersi Kriteri)

## Metodoloji

Bu simülasyon, belirli şikayet kategorilerini çözmenin oyuncu memnuniyetine ve
potansiyel gelire etkisini tahmin etmektedir.

**Varsayımlar:**
- Bir kategorinin tamamen çözülmesi, o kategoriye ait olumsuz yorumların %60'ını olumluya dönüştürür
- Olumsuz yorumdan olumluya dönen her 10 oyuncudan 1'i oyunu yeniden satın alır (churn recovery)
- Oyun fiyatı olarak meta veriden elde edilen medyan fiyat kullanılır

> Bu simülasyon oyun stüdyolarına geliştirme önceliği belirleme konusunda rehberlik eder.


In [ ]:
COZUM_ETKI_ORANI = 0.60
GERI_KAZANIM_ORANI = 0.10
TOPLAM_OYUNCU_HAVUZU = 1_000_000

if 'fiyat_usd' in df_merged.columns:
    ORTALAMA_FIYAT = df_merged['fiyat_usd'].replace(0, np.nan).median()
    if pd.isna(ORTALAMA_FIYAT):
        ORTALAMA_FIYAT = 29.99
else:
    ORTALAMA_FIYAT = 29.99

print(f'Ortalama oyun fiyati: ${ORTALAMA_FIYAT:.2f}')
print(f'Oyuncu havuzu: {TOPLAM_OYUNCU_HAVUZU:,}')


In [ ]:
def senaryo_hesapla(kategoriler, etiket):
    # Secili kategorilere sahip olumsuz yorum sayisi
    mask = df_merged['voted_up'] == False
    for kat in kategoriler:
        if kat in df_merged.columns:
            mask = mask & (df_merged[kat] == 1)

    etkilenen_yorum = mask.sum()
    toplam_olumsuz = (df_merged['voted_up'] == False).sum()
    toplam_yorum = len(df_merged)

    azalan_olumsuz = etkilenen_yorum * COZUM_ETKI_ORANI
    yeni_olumsuz = toplam_olumsuz - azalan_olumsuz
    eski_mem = (toplam_yorum - toplam_olumsuz) / toplam_yorum * 100
    yeni_mem = (toplam_yorum - yeni_olumsuz) / toplam_yorum * 100
    mem_artis = yeni_mem - eski_mem

    geri_kazanilan = azalan_olumsuz / toplam_yorum * TOPLAM_OYUNCU_HAVUZU * GERI_KAZANIM_ORANI
    ek_gelir = geri_kazanilan * ORTALAMA_FIYAT

    return {
        'Senaryo': etiket,
        'Etkilenen Yorum': int(etkilenen_yorum),
        'Memnuniyet Artisi (%)': round(mem_artis, 2),
        'Geri Kazanilan Oyuncu': int(geri_kazanilan),
        'Ek Gelir ($)': round(ek_gelir, 0),
    }


en_yuksek_kat = df_merged[kat_mevcut].sum().idxmax()
en_yuksek_iki = df_merged[kat_mevcut].sum().nlargest(2).index.tolist()

sonuclar = [
    senaryo_hesapla([en_yuksek_kat],  f'S1: {label_map.get(en_yuksek_kat, en_yuksek_kat)} Cozulurse'),
    senaryo_hesapla(en_yuksek_iki,    'S2: Ilk 2 Kategori Cozulurse'),
    senaryo_hesapla(kat_mevcut,       'S3: Tum Kategoriler Cozulurse'),
]

sim_df = pd.DataFrame(sonuclar)
print(sim_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

renkler = ['#3498db', '#e67e22', '#2ecc71']
metrikler = ['Memnuniyet Artisi (%)', 'Geri Kazanilan Oyuncu', 'Ek Gelir ($)']

for i, (metrik, ax) in enumerate(zip(metrikler, axes)):
    vals = sim_df[metrik]
    bars = ax.bar(range(3), vals, color=renkler, edgecolor='white')
    lbls = [f'{v:,.0f}' if metrik != 'Memnuniyet Artisi (%)' else f'{v:.2f}%' for v in vals]
    ax.bar_label(bars, labels=lbls, padding=3, fontsize=9)
    ax.set_xticks(range(3))
    ax.set_xticklabels(['S1', 'S2', 'S3'])
    ax.set_title(metrik, fontsize=11, fontweight='bold')

plt.suptitle('Maliyet/Fayda Simülasyonu Sonuclari', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
# Bölüm 8: SHAP Analizi (Açıklanabilir YZ)

## SHAP Nedir?

**SHAP (SHapley Additive exPlanations)**, oyun teorisinden ilham alan bir açıklanabilirlik yöntemidir.
Her özelliğin modelin tahminlerine katkısını bireysel düzeyde ölçer.

**Neden Kullandık?**
- Kara kutu modelleri şeffaf hale getirmek
- Oyuncu memnuniyetini (voted_up) etkileyen faktörleri anlamak
- Şikayet kategorilerinin önem sıralamasını ortaya koymak


In [ ]:
# Önce shap kurulumunu kontrol et
try:
    import shap
    print('shap mevcut, versiyon:', shap.__version__)
except ImportError:
    import subprocess, sys
    print('shap kuruluyor...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap
    print('shap kuruldu.')

_SHAP_OK = False
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import LabelEncoder
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report

    print('Mevcut sutunlar:', df_merged.columns.tolist())

    feature_cols = kat_mevcut.copy()

    # playtime_forever — NaN'ları 0 ile doldur
    if 'playtime_forever' in df_merged.columns:
        df_merged['playtime_forever'] = df_merged['playtime_forever'].fillna(0)
        feature_cols.append('playtime_forever')

    # language encode
    if 'language' in df_merged.columns:
        le = LabelEncoder()
        df_merged['language_enc'] = le.fit_transform(df_merged['language'].fillna('unknown'))
        feature_cols.append('language_enc')

    # Sadece sayısal sütunları seç ve NaN'ları 0 yap
    X_raw = df_merged[feature_cols].select_dtypes(include=[np.number]).fillna(0)
    feature_cols_clean = X_raw.columns.tolist()

    y_raw = df_merged['voted_up'].dropna()
    y_raw = y_raw.map({True: 1, False: 0,
                        'True': 1, 'False': 0,
                        1: 1, 0: 0}).dropna().astype(int)
    common_idx = X_raw.index.intersection(y_raw.index)
    X = X_raw.loc[common_idx]
    y = y_raw.loc[common_idx]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    from sklearn.metrics import accuracy_score

    rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    print('Model egitimi tamamlandi.')
    print(classification_report(y_test, rf.predict(X_test),
                                target_names=['Olumsuz', 'Olumlu']))
    print('Model egitildi, dogruluk:', accuracy_score(y_test, rf.predict(X_test)))
    print('SHAP degerleri hesaplaniyor...')
    print('X_test shape:', X_test.shape)
    _SHAP_OK = True

except Exception as e:
    print('Hata:', e)
    _SHAP_OK = False


## 8.1 SHAP Summary Plot


In [ ]:
import matplotlib.pyplot as plt

if _SHAP_OK:
    try:
        import os
        os.makedirs('reports/figures', exist_ok=True)

        explainer = shap.TreeExplainer(rf)
        shap_values = explainer.shap_values(X_test)

        sv_plot = shap_values[1] if isinstance(shap_values, list) else shap_values

        plt.figure(figsize=(10, 6))
        shap.summary_plot(sv_plot, X_test,
                          feature_names=X_test.columns.tolist(),
                          show=False)
        plt.title('SHAP Summary Plot - Özellik Etki Analizi')
        plt.tight_layout()
        plt.savefig('reports/figures/shap_summary.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        print('SHAP Summary Plot kaydedildi.')

        _shap_values = shap_values
    except Exception as e:
        print('Hata:', e)


## 8.2 SHAP Bar Plot — Özellik Önem Sıralaması


In [ ]:
import matplotlib.pyplot as plt

if _SHAP_OK:
    try:
        import os
        os.makedirs('reports/figures', exist_ok=True)

        sv_plot = _shap_values[1] if isinstance(_shap_values, list) else _shap_values

        plt.figure(figsize=(10, 6))
        shap.summary_plot(sv_plot, X_test,
                          feature_names=X_test.columns.tolist(),
                          plot_type='bar',
                          show=False)
        plt.title('SHAP Bar Plot - Özellik Önem Siralaması')
        plt.tight_layout()
        plt.savefig('reports/figures/shap_bar.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        print('SHAP Bar Plot kaydedildi.')
    except Exception as e:
        print('Hata:', e)


**Yorum:** Bu analiz hangi faktörlerin oyuncu memnuniyetini en çok etkilediğini göstermektedir.
SHAP değerleri sıfırın sağında olan özellikler olumlu oylamayı artırıcı, solunda olanlar azaltıcı etkiye sahiptir.


---
# Bölüm 9: Bulgular ve Sonuçlar


## 9.1 Oyun Türüne Göre Özet Tablo


In [ ]:
try:
    print('Mevcut sutunlar:', df_merged.columns.tolist())

    if 'game_genre' not in df_merged.columns:
        print('UYARI: game_genre sütunu bulunamadı.')
    else:
        ozet_ops = {}
        ozet_ops['yorum_sayisi'] = ('game_name', 'count')

        if 'voted_up' in df_merged.columns:
            ozet_ops['olumlu_pct'] = ('voted_up', lambda x: (x == True).mean() * 100)

        vader_col_ozet = next((c for c in ['vader_compound', 'compound', 'vader_score']
                               if c in df_merged.columns), None)
        if vader_col_ozet:
            ozet_ops['ort_vader'] = (vader_col_ozet, 'mean')

        if 'playtime_forever' in df_merged.columns:
            ozet_ops['ort_playtime_saat'] = ('playtime_forever', lambda x: x.mean() / 60)

        if 'mixed_sentiment' in df_merged.columns:
            ozet_ops['mixed_pct'] = ('mixed_sentiment', 'mean')

        ozet = df_merged.groupby('game_genre').agg(**ozet_ops).round(2)
        print('=== OYUN TÜRÜ ÖZET İSTATİSTİKLERİ ===')
        print(ozet.to_string())
except Exception as e:
    print('Hata:', e)


## 9.2 En Önemli 5 Bulgu

1. **Teknik sorunlar ve performans**, tüm türlerde en yaygın şikayet kategorileridir.
   FPS ve çok oyunculu oyunlarda bu oran özellikle yüksektir.

2. **Mixed sentiment** (olumlu oy + negatif yorum) oranı bazı oyunlarda %20'yi aşmaktadır;
   bu oyunlar yüksek potansiyelli ama iyileştirme gerektiren ürünlerdir.

3. **Deneyimli oyuncular** (600+ saat) performans ve içerik şikayetlerini daha şiddetli dile getirmektedir;
   ağırlıklı skor bu farkı ortaya koymaktadır.

4. **Fiyat ile şikayet yoğunluğu** arasındaki korelasyon zayıftır — pahalı oyunlar illa ki
   daha fazla şikayet almamaktadır. Ücretsiz oyunlar (F2P) fiyat/monetizasyon şikayetlerinde öne çıkmaktadır.

5. **SHAP analizi**, oyuncu memnuniyetini en çok etkileyen faktörün
   teknik sorunlar ve oynanabilirlik olduğunu göstermektedir.


## 9.3 Oyun Stüdyolarına Öneriler


In [ ]:
print('=== OYUN STÜDYOLARI İÇİN ÖNERİLER ===')
print()

oneriler = [
    ('FPS Oyunlar',       'Sunucu kararliligi ve hile önleme sistemlerine yatirim yapin.'),
    ('RPG Oyunlar',       'Teknik hatalari erken yamalarla giderin; hikaye tutarliligina önem verin.'),
    ('Strateji Oyunlar',  'Icerik güncellemelerini düzenli tutun; denge sorunlarini hizli ele alin.'),
    ('Simulasyon Oyunlar','DLC fiyatlandirmasini ve icerik miktarini seffaf tutun.'),
    ('Cok Oyunculu',      'Matchmaking ve topluluk toksikligi sorunlarini önceliklendirin.'),
]

for tur, oneri in oneriler:
    print(f'  [{tur}]')
    print(f'    -> {oneri}')
    print()


## 9.4 Çalışmanın Kısıtları ve Gelecek Çalışmalar

### Kısıtlar
- Her oyundan 500 yorum örneklenmiştir; tam veri daha güçlü sonuçlar verebilir
- VADER İngilizce için optimize edilmiştir; çok dilli yorumlar çeviri gerektirir
- Kategorizasyon anahtar kelime tabanlıdır; bağlam duyarlı NLP modelleri daha iyi sonuç verebilir
- Fiyat verisi anlık değerdir; indirim dönemleri yansıtılmamıştır

### Gelecek Çalışmalar
- BERT/RoBERTa tabanlı duygu analizi
- LDA/BERTopic ile konu modelleme
- Zaman serisi anomali tespiti (güncelleme sonrası şikayet artışı)
- Oyuncu segmentasyonu (playtime bazlı clustering)
- Steam dışı platformlarla (Reddit, Metacritic) veri zenginleştirme

---
*Bu analiz Python dersi projesi kapsamında hazırlanmıştır. Tüm veriler Steam API üzerinden yasal yollarla toplanmıştır.*
